In [30]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

DATA_PATH = "train_ver2.csv"  
LAST_DATE = "2016-05-28"

header = pd.read_csv(DATA_PATH, nrows=0).columns
prod_cols = list(header[24:48])
print(len(prod_cols), prod_cols[:3], prod_cols[-1])   

24 ['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1'] ind_recibo_ult1


In [31]:
use_q1 = ["fecha_dato", "ncodpers", "nomprov", "renta", "age"]

chunks = []
for chunk in pd.read_csv(DATA_PATH, usecols=use_q1, dtype=str, chunksize=500_000):
    chunks.append(chunk[chunk["fecha_dato"] == LAST_DATE])
last = pd.concat(chunks, ignore_index=True)

last["age"] = pd.to_numeric(last["age"], errors="coerce")
last["renta"] = pd.to_numeric(last["renta"], errors="coerce")

g = last.groupby("nomprov")

city_stats = pd.DataFrame({
    "고객수": g["ncodpers"].count(),
    "평균소득": g["renta"].mean(),
    "소득중앙값": g["renta"].median(),
    "평균나이": g["age"].mean(),
})

top5 = (city_stats[city_stats["고객수"] >= 50]
        .sort_values("평균소득", ascending=False)
        .head(5))
top5

,고객수,평균소득,소득중앙값,평균나이
nomprov,,,,
CEUTA,513,200515.885152,128699.580,44.007797
MADRID,298841,178362.993637,138024.135,45.205564
"BALEARS, ILLES",9143,171890.951564,122837.220,39.748770
BARCELONA,88754,164584.843157,130877.160,39.865144
MELILLA,641,149071.944149,117718.365,45.329173


In [32]:
use_q2 = ["fecha_dato", "ncodpers"] + prod_cols
dtypes = {"ncodpers": "int32", **{c: "float32" for c in prod_cols}}

df = pd.read_csv(DATA_PATH, usecols=use_q2, dtype=dtypes, parse_dates=["fecha_dato"])

FILL_NA = True
REQUIRE_CONSECUTIVE = True

if FILL_NA:
    df[prod_cols] = df[prod_cols].fillna(0)

df["month"] = (df["fecha_dato"].dt.year * 12 + df["fecha_dato"].dt.month).astype("int32")
df = df.sort_values(["ncodpers", "month"]).reset_index(drop=True)

same_cust = df["ncodpers"].eq(df["ncodpers"].shift(1))
if REQUIRE_CONSECUTIVE:
    valid = same_cust & ((df["month"] - df["month"].shift(1)) == 1)
else:
    valid = same_cust

monthly_new = {}
for c in prod_cols:
    prev = df[c].shift(1)
    new_flag = valid & (df[c] == 1) & (prev == 0)
    monthly_new[c] = new_flag.groupby(df["fecha_dato"]).sum()

monthly_new = pd.DataFrame(monthly_new).astype(int)
monthly_new = monthly_new.iloc[1:]
monthly_new.columns.name = "product"
monthly_new.index.name = "fecha_dato"

total_new = monthly_new.sum().sort_values(ascending=False)
total_new.name = None

print("Q2: 신규 가입이 가장 많은 상품 5개")
print("전체 기간 신규 가입 건수")
print(total_new.head(5))

print("\n월별·상품별 신규 가입 건수 (행: 월, 열: 상품)")
monthly_new

Q2: 신규 가입이 가장 많은 상품 5개
전체 기간 신규 가입 건수
product
ind_recibo_ult1      153146
ind_nom_pens_ult1     84709
ind_nomina_ult1       73753
ind_tjcr_fin_ult1     69118
ind_cco_fin_ult1      69103
dtype: int64

월별·상품별 신규 가입 건수 (행: 월, 열: 상품)


product,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
fecha_dato,,,,,,,,,,,,,,,,,,,,,
2015-02-28,1,1,2132,12,1884,7,291,249,171,348,...,5,27,9,217,3471,259,8,5659,5656,8285
2015-03-28,0,1,2293,15,1786,18,272,284,165,481,...,7,24,12,217,5076,205,4,4479,4473,9135
2015-04-28,0,1,2366,12,1984,10,217,278,167,263,...,4,22,17,871,4627,194,5,2663,5628,8346
2015-05-28,0,0,2264,7,1900,10,205,289,172,238,...,9,21,11,343,4071,201,4,3369,3345,7085
2015-06-28,0,0,6588,9,1857,7,210,213,151,289,...,4,19,8,2931,4655,152,3,5070,8135,9023
2015-07-28,0,0,3056,10,2771,18,191,252,164,265,...,6,27,5,589,4886,232,9,4408,4850,9172
2015-08-28,0,0,5730,10,2412,37,240,231,145,322,...,1,22,13,264,4379,445,2,2950,2968,7115
2015-09-28,0,0,4389,7,1775,22,471,191,122,387,...,8,21,13,271,4463,328,2,4683,4753,11210
2015-10-28,0,0,5682,3,2409,43,468,223,153,349,...,7,19,9,593,4461,911,2,4818,4790,12100


In [33]:
use_q3 = ["fecha_dato", "sexo", "age", "segmento", "ind_empleado", "tiprel_1mes"] + prod_cols

chunks = []
for chunk in pd.read_csv(DATA_PATH, usecols=use_q3, dtype=str, chunksize=500_000):
    chunks.append(chunk[chunk["fecha_dato"] == LAST_DATE])
d = pd.concat(chunks, ignore_index=True)

d[prod_cols] = d[prod_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
d["age"] = pd.to_numeric(d["age"], errors="coerce")

d["n_prod"] = d[prod_cols].sum(axis=1)
d["target"] = np.where(d["n_prod"] >= 3, "3개 이상", "3개 미만")

d["age_group"] = np.select(
    [d["age"] >= 65, d["age"] >= 30, d["age"] < 30],
    ["senior", "adult", "young"],
    default=None,
)

candidates = {
    "1) 성별 (sexo)": "sexo",
    "2) 나이그룹 (age_group)": "age_group",
    "3) segmento": "segmento",
    "4) ind_empleado": "ind_empleado",
    "5) tiprel_1mes": "tiprel_1mes",
}

rows = []
for name, col in candidates.items():
    sub = d[[col, "target"]].dropna()
    sub[col] = sub[col].str.strip()
    sub = sub[sub[col] != ""]
    ct = pd.crosstab(sub[col], sub["target"])
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))   
    rows.append({"기준": name, "chi2": chi2, "p-value": p, "dof": dof, "Cramér's V": cramers_v})

result = pd.DataFrame(rows).sort_values("chi2", ascending=False).reset_index(drop=True)
result

,기준,chi2,p-value,dof,Cramér's V
0,5) tiprel_1mes,183808.591523,0.000000e+00,3,0.445184
1,3) segmento,124374.548799,0.000000e+00,2,0.366800
2,2) 나이그룹 (age_group),61002.202414,0.000000e+00,2,0.255913
3,1) 성별 (sexo),3412.231367,0.000000e+00,1,0.060526
4,4) ind_empleado,1343.689088,1.120822e-289,4,0.037981
